In [1]:
# From Bob: I uncommented this line or else I cannot query the data in block 3.
from project import icu_query, vitals_query

In [2]:
from project import bucket_ecg_report_0, simplify_race, simplify_careunit
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df_icu = icu_query()
df_vitals = vitals_query()
original_query = df_icu.merge(df_vitals, on='stay_id', how='left')

/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
df = original_query
df.head()

,subject_id_x,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,mbp,sbp_ni,dbp_ni,mbp_ni,resp_rate,temperature,temperature_site,spo2,glucose,rn
0,19942060,26995122,32178416,Neurology,Neurology,2161-01-11 16:09:00,2161-02-08 22:05:32,28.247593,2161-01-11 15:11:00,2161-02-10 13:41:00,...,72.0,115.0,58.0,72.0,40.0,36.610000000,Oral,94.0,NaN,1
1,17945971,25486527,36652195,Med/Surg,Med/Surg,2130-03-17 04:25:01,2130-03-18 15:04:34,1.444132,2130-03-16 19:53:00,2130-03-20 16:10:00,...,NaN,NaN,NaN,NaN,18.0,None,None,NaN,NaN,1
2,13570398,29201840,30059691,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2182-10-19 17:28:12,2182-10-27 20:38:37,8.132234,2182-10-18 12:17:00,2182-10-28 14:10:00,...,NaN,NaN,NaN,NaN,NaN,None,None,NaN,129.0,1
3,12506390,21301912,30062923,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2160-10-17 13:39:06,2160-10-18 14:19:00,1.027708,2160-10-16 10:30:00,2160-10-25 14:05:00,...,NaN,NaN,NaN,NaN,NaN,None,None,NaN,175.0,1
4,14178148,28718602,30126048,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2142-09-14 03:01:44,2142-09-16 19:13:11,2.674618,2142-09-13 22:53:00,2142-09-16 15:18:00,...,105.0,139.0,96.0,105.0,NaN,None,None,NaN,NaN,1


In [ ]:
# need to bucket report_0...
# getting frequency of report_0 codes
original_ecg_report0_frequency = original_query['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')

In [ ]:
# sinus rhythm, sinus tachycardia, a.fib
df['report_0'] = df['report_0'].str.lower().str.removesuffix(".").str.replace('"', '')
df = df[~df['report_0'].str.contains('warning')]
initial_drop = df['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model
df = df[df['report_0'].notna()] # handles any NA values

In [ ]:
# check if any NAs, there were none
print(original_query.shape)
print(df.shape)

In [ ]:
# check mortality captured by these first four catgories ()
df_first_cats = df[df['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', 'sinus bradycardia'])]
print(f"")
print(f"Mortalities captured by the first four report_0 buckets: {sum(df_first_cats['hospital_expire_flag'])} ({sum(df_first_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")

In [ ]:
df['ecg_bucket'] = df['report_0'].apply(bucket_ecg_report_0)
# check mortality captured after bucket_ecg_report_0 function
df_cats = df[df['ecg_bucket'].isin(['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional'])]
print(f"Mortalities captured by the current buckets: {sum(df_cats['hospital_expire_flag'])} ({sum(df_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")
post_bucket_function = df['ecg_bucket'].value_counts()
post_bucket_function.to_csv('sanity_outputs/post_bucket_function.csv')
# capturing 98.7% of mortalities in all categories != 'other'

In [ ]:
named_buckets = ['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional']
df_other = df[~df['ecg_bucket'].isin(named_buckets)]
df_other.groupby('ecg_bucket')['hospital_expire_flag'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)

In [ ]:
df['ecg_bucket'] = df['ecg_bucket'].where(df['ecg_bucket'].isin(named_buckets), 'other')
df['ecg_bucket'].value_counts()

In [ ]:
df.to_csv('sanity_outputs/report0_cleaned.csv')
# report_0 cleaned values in new column: ecg_bucket

In [ ]:
# set gender to 0 (Male), 1 (Female)
df['gender'] = df['gender'].replace(['M', 'F'], [0,1])

In [ ]:
# cleaning ecg measurements: explore columns
ecg_intervals = ['rr_interval','p_onset', 'p_end', 'qrs_onset', 'qrs_end', 't_end']
ecg_axes = ['p_axis', 'qrs_axis', 't_axis']

display(df[ecg_intervals].describe())
display(df[ecg_axes].describe())

In [ ]:
# drop p_onset, p_end, p_axis as predictor because they have so many values at 29999 ms (physiologically impossible) imputing would be dangerous
df = df.drop(columns=['p_onset', 'p_end', 'p_axis'])

# imputing impossible values with median of associated ecg_bucket value
for col in ['rr_interval', 'qrs_onset', 'qrs_end', 't_end']:
    df.loc[df[col]>4000, col] = pd.NA
for col in ['qrs_axis', 't_axis']:
    df.loc[~df[col].between(-180,180), col] = pd.NA

ecg_measurements = ['rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis']
df[ecg_measurements]=df[ecg_measurements].astype(float)
df[ecg_measurements] = df.groupby('ecg_bucket')[ecg_measurements].transform(lambda x: x.fillna(x.median()))

In [ ]:
# handle missing values: language, marital status
# handle spaces and case: language, marital status, race, admission type, admission location, first care unit, last care unit
df['language'] = df['language'].fillna('unknown')
df['language'] = df['language'].str.lower().str.replace(' ', '_')

df['marital_status'] = df['marital_status'].fillna('unknown')
df['marital_status'] = df['marital_status'].str.lower()

df['race'] = df['race'].apply(simplify_race) # function handles

df['admission_type'] = df['admission_type'].str.lower().str.replace(' ','_').str.removesuffix('.')

df['admission_location'] = df['admission_location'].str.lower().str.replace('-', '_').str.replace(' ', '_').str.replace('/', '_')
df['admission_location'].value_counts()

# there are no differences in facilities between first vs last care unit in this dataset, shrinking to 1 column: care_unit
df = df.rename(columns={'first_careunit': 'care_unit'})
df['care_unit'] = df['care_unit'].apply(simplify_careunit)
df = df.drop(columns=['last_careunit'])

# change 'anchor_age' col to 'age'
df = df.rename(columns={'anchor_age': 'age'})

In [ ]:
# drop columns that will not be used in modeling
keep_cols = [
    'care_unit', 'admission_type', 'admission_location', 
    'language', 'marital_status', 'race', 'hospital_expire_flag', 
    'gender', 'age', 'rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis', 'ecg_bucket'
]
df = df[keep_cols]
df.shape

In [ ]:
# Bob: Check columns, head, and export cleaned data to csv
print(df.columns)
display(df.head())
df.to_csv('data/cleaned_data.csv')